This notebook aims at fetching CMIP6 climate data from pangeo API and store as zarr files for one scenario / experiment.
The scenarios / experiments are available in scenario_experiment_combination.json.

In [1]:
import json
from dask.distributed import Client
import intake

import ipywidgets as widgets
from IPython.display import display

from src.data.cmip6_pangeo import get_cmip6_data_from_pangeo_api

In [2]:
# Start Dask Client
client = Client(n_workers=1, threads_per_worker=4, memory_limit="4GB")
print(f"Dask dashboard: {client.dashboard_link}")


url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
col = intake.open_esm_datastore(url)
z_kwargs = {"consolidated": True, "decode_times": True, "use_cftime": True}

with open("./scenario_experiment_combination.json", "r") as f:
    models = json.load(f)

tbl_var = {
    "Amon": [
        "hurs",
        "psl",
        "ta",
    ],
    "Omon": ["tos"],
}
store_dir = "./data/input/cmip6_data"
print(f"Save dir: {store_dir}")

Dask dashboard: http://127.0.0.1:8787/status


/home/tiphanie/projets_python/catherina/.pixi/envs/default/lib/python3.13/site-packages/intake_esm/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Save dir: ./data/input/cmip6_data


Choose the scenario and experiment

In [3]:
source_dropdown = widgets.Dropdown(
    options=[m["source_id"] for m in models],
    description="Model:"
)

experiment_dropdown = widgets.Dropdown(
    description="Experiment:"
)


def update_experiments(change):
    selected_source = change["new"]
    model = next(m for m in models if m["source_id"] == selected_source)

    experiment_dropdown.options = [
        "-- Select an experiment --",
        *model["experiments"]
    ]
    experiment_dropdown.value = "-- Select an experiment --"

source_dropdown.observe(update_experiments, names="value")

update_experiments({"new": source_dropdown.value})
display(source_dropdown, experiment_dropdown)

Dropdown(description='Model:', options=('ACCESS-CM2', 'MPI-ESM1-2-LR', 'FGOALS-g3', 'IPSL-CM6A-LR', 'UKESM1-0-…

Dropdown(description='Experiment:', options=('-- Select an experiment --', 'historical', 'ssp245', 'ssp370', '…

Attention : à date, les scénarios MIROC et UKESM10L ne fonctionnent pas. 

In [5]:
source_id = source_dropdown.value
experiment_id = experiment_dropdown.value

if experiment_id == "-- Select an experiment --":
    print("Please select a model and an experiment.")
else:
    print(f"Selected model: {source_id}, experiment: {experiment_id}")

    # Find the corresponding entry in the JSON
    model_info = next(
        m for m in models
        if m["source_id"] == source_id
    )

    # Retrieve all parameters
    institution_id = model_info["institution_id"]
    member_id = model_info["member_id"]

    print(f"institution_id : {institution_id}")
    print(f"member_id      : {member_id}")

    # Retrieve cmip6 data
    get_cmip6_data_from_pangeo_api(
        source_id=source_id,
        experiment_id=experiment_id,
        store_dir=store_dir,
        institution_id=institution_id,
        member_id=member_id,
        col=col,
        tbl_var=tbl_var,
        z_kwargs=z_kwargs
    )

Selected model: ACCESS-CM2, experiment: historical
institution_id : CSIRO-ARCCSS
member_id      : r1i1p1f1

Sample query:
{'experiment_id': 'historical',
 'table_id': 'Amon',
 'source_id': 'ACCESS-CM2',
 'variable_id': 'hurs',
 'member_id': 'r1i1p1f1',
 'grid_label': ['gn'],
 'institution_id': 'CSIRO-ARCCSS'}


Processing pangeo queries:   0%|          | 0/4 [00:00<?, ?it/s]

{'experiment_id': 'historical',
 'table_id': 'Amon',
 'source_id': 'ACCESS-CM2',
 'variable_id': 'hurs',
 'member_id': 'r1i1p1f1',
 'grid_label': ['gn'],
 'institution_id': 'CSIRO-ARCCSS'}

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


/home/tiphanie/projets_python/catherina/.pixi/envs/default/lib/python3.13/site-packages/intake_esm/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Processing model-experiment-variable:   0%|          | 0/1 [00:00<?, ?it/s]

...

Coordinates:
             CF Axes: * X: ['x']
                        Y: ['lat', 'y']
                      * T: ['time']
                        Z: n/a

      CF Coordinates: * longitude: ['x']
                        latitude: ['lat', 'y']
                      * time: ['time']
                        vertical: n/a

       Cell Measures:   area, volume: n/a

      Standard Names:   latitude: ['lat', 'y']
                      * longitude: ['x']
                      * time: ['time']

              Bounds:   n/a

       Grid Mappings:   n/a

Data Variables:
       Cell Measures:   area, volume: n/a

      Standard Names:   air_temperature: ['ta']
                        latitude: ['lon_b']
                        longitude: ['lat_b']

              Bounds:   Y: ['lat_b']
                        lat: ['lat_b']
                        latitude: ['lat_b']
                        lon: ['lon_b']

       Grid Mappings:   n/a


/home/tiphanie/projets_python/catherina/.pixi/envs/default/lib/python3.13/site-packages/distributed/client.py:3371: UserWarning: Sending large graph of size 46.94 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


{'experiment_id': 'historical',
 'table_id': 'Omon',
 'source_id': 'ACCESS-CM2',
 'variable_id': 'tos',
 'member_id': 'r1i1p1f1',
 'grid_label': ['gn'],
 'institution_id': 'CSIRO-ARCCSS'}

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


Processing model-experiment-variable:   0%|          | 0/1 [00:00<?, ?it/s]

/home/tiphanie/projets_python/catherina/.pixi/envs/default/lib/python3.13/site-packages/distributed/client.py:3371: UserWarning: Sending large graph of size 50.75 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
